In [1]:
import sys
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
# Agregar el directorio raíz al path de Python para que reconozca 'src'
root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
  sys.path.append(str(root_dir))

from src.pipeline import run_pipeline

# Ejecutar el pipeline apuntando a la carpeta sample local
print("Ejecutando pipeline de prueba con el sample...")
counts = run_pipeline(
    sources_url=None,
    token=None,
    output_dir=root_dir / "output",
    sample_dir=root_dir / "sample",
)
print("¡Pipeline terminado con éxito! Conteo:", counts)

Ejecutando pipeline de prueba con el sample...


INFO:pipeline:Ingesting Clientes...
INFO:pipeline:Ingesting Empleados...
INFO:pipeline:Ingesting Slack Users...
INFO:pipeline:Processing tables...
INFO:pipeline:Processing docs...
INFO:pipeline:Processing chats...
INFO:pipeline:Ejecutando guardrail de validación anti-fugas...
ERROR:pipeline:¡FUGA DETECTADA! El valor original 'ldiaz' (Entity: p_069) sigue presente en el registro clientes:CL-72856
ERROR:pipeline:¡FUGA DETECTADA! El valor original 'Verónica' (Entity: p_088) sigue presente en el registro 09f5ec34-bd6b-5b30-96c4-67aecdbdae34
ERROR:pipeline:¡FUGA DETECTADA! El valor original 'Verónica' (Entity: p_088) sigue presente en el registro bdde8eb9-65e1-5e72-a62a-de5618195dcc
ERROR:pipeline:¡FUGA DETECTADA! El valor original 'Verónica' (Entity: p_088) sigue presente en el registro 9e0e0572-3746-5a1a-ba83-560a64bdddbe
ERROR:pipeline:¡FUGA DETECTADA! El valor original 'Verónica' (Entity: p_088) sigue presente en el registro 2d493256-cf41-55e8-9e5f-c4367f0d23f1
ERROR:pipeline:¡FUGA DETE

RuntimeError: ¡El pipeline se detuvo por seguridad! Se detectaron 8 fugas de PII en la salida.

# Conexión Tablas (Schema)

In [ ]:
# Definir la ruta de la base de datos basada en la configuración del proyecto
root_dir = Path().resolve().parent
db_path = root_dir / "data" / "entity_map.db"

print(f"¿Existe la base de datos en {db_path}?: {db_path.exists()}")

# Conectar y consultar las tablas del schema
conn = sqlite3.connect(str(db_path))
query_tables = "SELECT name, sql FROM sqlite_master WHERE type='table';"
df_tables = pd.read_sql_query(query_tables, conn)

print("\n--- SCHEMA DE LA BASE DE DATOS ---")
for idx, row in df_tables.iterrows():
  print(f"\n📦 Tabla: {row['name']}")
  print(row["sql"])

conn.close()

¿Existe la base de datos en /Users/audeemii/Desktop/frontier-hackathon-2026/data/entity_map.db?: True

--- SCHEMA DE LA BASE DE DATOS ---

📦 Tabla: entities
CREATE TABLE entities (
                    entity_id TEXT PRIMARY KEY,
                    type TEXT NOT NULL,
                    gender TEXT,
                    name TEXT,
                    first_name TEXT,
                    email TEXT,
                    phone TEXT,
                    cedula TEXT,
                    address TEXT,
                    birth_iso TEXT,
                    plate TEXT,
                    passport TEXT,
                    account TEXT,
                    handle TEXT,
                    kept INTEGER DEFAULT 0,
                    reason TEXT
                )

📦 Tabla: entity_originals
CREATE TABLE entity_originals (
                    entity_id TEXT NOT NULL,
                    original_value TEXT NOT NULL,
                    val_type TEXT NOT NULL,
                    PRIMARY KEY (enti

# Inspeccionar la Tabla de Entidades (entities)

In [ ]:
conn = sqlite3.connect(str(db_path))
total_ent = pd.read_sql_query(
    "SELECT COUNT(*) as total FROM entities;", conn
).iloc[0]["total"]
print(f"Total de entidades registradas: {total_ent}")

df_entities = pd.read_sql_query("SELECT * FROM entities LIMIT 10;", conn)
conn.close()

display(df_entities)

Total de entidades registradas: 81


,entity_id,type,gender,name,first_name,email,phone,cedula,address,birth_iso,plate,passport,account,handle,kept,reason
0,kept_001,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
1,kept_002,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
2,kept_003,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
3,kept_004,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
4,kept_005,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
5,kept_006,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
6,kept_007,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
7,kept_008,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
8,kept_009,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."
9,kept_010,organization_or_place,None,None,None,None,None,None,None,None,None,None,None,None,1,"organización, lugar, marca o figura pública; n..."


# Inspeccionar los Mapeos de Originales (entity_originals)

In [ ]:
conn = sqlite3.connect(str(db_path))
df_originals = pd.read_sql_query(
    "SELECT * FROM entity_originals LIMIT 15;", conn
)
conn.close()

display(df_originals)

,entity_id,original_value,val_type
0,kept_001,Bancolombia,organization
1,kept_002,Superfinanciera,organization
2,kept_003,Nequi,organization
3,kept_004,Daviplata,organization
4,kept_005,Davivienda,organization
5,kept_006,Banco de Bogotá,organization
6,kept_007,Banco de Occidente,organization
7,kept_008,Santander,organization
8,kept_009,BBVA,organization
9,kept_010,PSE,organization


# Inspeccionar el Estado de los Jobs (jobs)

In [ ]:
conn = sqlite3.connect(str(db_path))
df_jobs = pd.read_sql_query("SELECT * FROM jobs;", conn)
conn.close()

display(df_jobs)

,job_id,status,started_at,finished_at,chats_count,docs_count,tables_count,tokens_in,tokens_out,seconds,error


# Inspeccionar personas (kept=0)

In [ ]:
conn = sqlite3.connect(str(db_path))
# Consultar registros donde 'kept' sea 0 (personas naturales anonimizadas)
df_people = pd.read_sql_query(
    "SELECT entity_id, name, email, phone, cedula FROM entities WHERE kept = 0 LIMIT 10;",
    conn,
)
conn.close()

display(df_people)

,entity_id,name,email,phone,cedula
0,p_001,Lucía Rodríguez,lrodriguez@hotmail.com,3112372053,1001374921
1,p_002,Liliana Díaz,ldiaz@outlook.com,3122742106,1002749842
2,p_003,Gloria Romero,gromero@icloud.com,3103112159,1004124763
3,p_004,Gonzalo Pineda,gpineda@yahoo.com,3113482212,1005499684
4,p_005,Felipe González,felipe.gonzalez@kipupay.co,3123852265,1006874605
5,p_006,Gabriel Gutiérrez,gabriel.gutierrez@kipupay.co,3104222318,1008249526
6,p_007,Marcela Vega,marcela.vega@kipupay.co,3114592371,1009624447
7,p_008,Ximena Hoyos,ximena.hoyos@kipupay.co,3124962424,1010999368
8,p_009,Mateo Torres,mateo.torres@kipupay.co,3105332477,1012374289
9,p_010,Natalia Silva,natalia.silva@kipupay.co,3115702530,1013749210
